# 🚀 Fine-Tuning Sarvam-2B on Telugu & Tenglish Dialogue Corpus (LoRA / QLoRA)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaiKiranKumarNalla/tenglish-lm/blob/main/notebooks/03_FineTune_Sarvam_Telugu_Tenglish.ipynb)

**Features:**
- **Zero-Leakage Guarantee:** Strictly partitions 76 movies into 60 Train (12,336 scenes), 8 Validation (1,629 scenes), and 8 Test (1,632 scenes).
- **Dual Script Mastery:** Learns both native Telugu script and 3 Tenglish formats (P1 Natural Loanword-Preserved, P2 Phonetic, P3 Chat).
- **Memory Efficient:** Uses QLoRA (4-bit BitsAndBytes NF4 + PEFT) runnable on a free Google Colab T4 GPU (~5.5 GB VRAM).

In [ ]:
# @title Step 1: Install Training Dependencies (TRL, PEFT, Accelerate, BitsAndBytes)
!pip install -q transformers peft trl accelerate bitsandbytes datasets

In [ ]:
# @title Step 2: Clone Clean Dataset Partitions from GitHub
!git clone https://github.com/SaiKiranKumarNalla/tenglish-lm.git /content/tenglish-lm
%cd /content/tenglish-lm

import json
print("Loading split manifest...")
# Dataset manifests and JSONL splits
!ls -la data/splits/ || true

In [ ]:
# @title Step 3: Load Sarvam-2B in 4-bit QLoRA
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "sarvamai/sarvam-2b"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

# LoRA Configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("✅ Sarvam-2B QLoRA adapter configured and ready!")

In [ ]:
# @title Step 4: Format Training Data (Instruction Tuning for Dialogue & Tenglish)
from datasets import load_dataset

# We train on two high-value tasks:
# Task 1: Context-Conditioned Dialogue Continuation (Telugu & Tenglish)
# Task 2: Bidirectional Telugu Script <-> Tenglish Translation

def format_dialogue_prompt(item):
    # Formats scene context + ground truth next turn
    m = item.get('movie_name', '')
    d = item.get('director', '')
    emo = item.get('scene_emotion', 'CONVERSATIONAL_DRAMA')
    dial = item.get('scene_dialect', 'STANDARD_TELUGU')
    turns = item.get('turns_telugu', [])
    if len(turns) < 2:
        return None
    
    prompt = f"[సినిమా: {m}] [దర్శకత్వం: {d}] [భావోద్వేగం: {emo}] [మాండలికం: {dial}]\n\nసంభాషణ:\n"
    for idx, t in enumerate(turns[:-1], start=1):
        prompt += f"పాత్ర {idx}: {t}\n"
    prompt += f"పాత్ర {len(turns)}: {turns[-1]}"
    return {"text": prompt}

print("Data formatting pipeline initialized!")

In [ ]:
# @title Step 5: Start Training via SFTTrainer (TRL)
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./sarvam_telugu_tenglish_lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=200,
    num_train_epochs=3,
    warmup_ratio=0.03,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none"
)

print("Training configuration ready. Run trainer.train() to begin fine-tuning!")